# Standard Autoencoder Training — Google Colab GPU

This notebook trains only the existing convolutional Autoencoder. It downloads CASIA v2.0 directly into the Colab runtime and reuses the committed split manifests and Python modules. It intentionally does not perform final test metrics or denoising.

Before running, select **Runtime → Change runtime type → T4 GPU** (or another GPU).

In [1]:
# Colab already provides a CUDA-enabled PyTorch build. Do not reinstall torch.
%pip install -q kagglehub Pillow matplotlib numpy

In [2]:
import torch

cuda_available = torch.cuda.is_available()
print("CUDA available:", cuda_available)
assert cuda_available, "Enable a GPU runtime before continuing."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)

CUDA available: True
GPU: Tesla T4


## Obtain the project code

The large CASIA dataset is not stored in GitHub. Only source code and portable split CSVs are cloned.

In [3]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/chetanraje27/Digital-Evidence-GenAI.git"
PROJECT_ROOT = Path("/content/Digital-Evidence-GenAI")

if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"Updating existing checkout: {PROJECT_ROOT}")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)

os.chdir(PROJECT_ROOT)
print("Project root:", Path.cwd())

Project root: /content/Digital-Evidence-GenAI


## Download CASIA v2.0 with KaggleHub

This public Kaggle dataset is downloaded into `data/raw`. If Kaggle asks for authentication, add your Kaggle credentials to the Colab session and rerun this cell.

In [4]:
import kagglehub

DATASET_HANDLE = "divg07/casia-20-image-tampering-detection-dataset"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
downloaded_path = kagglehub.dataset_download(
    DATASET_HANDLE, output_dir=str(RAW_DATA_DIR)
)
print("Dataset location:", downloaded_path)
print("Authentic directory exists:", (RAW_DATA_DIR / "CASIA2" / "Au").is_dir())
print("Tampered directory exists:", (RAW_DATA_DIR / "CASIA2" / "Tp").is_dir())

100%|██████████| 2.56G/2.56G [01:03<00:00, 43.2MB/s]

Extracting files...


Dataset location: /content/Digital-Evidence-GenAI/data/raw
Authentic directory exists: True
Tampered directory exists: True


## Validate portable split manifests

The manifests contain repository-relative paths. This cell rejects absolute Windows/Linux paths, duplicates, missing files, and ground-truth mask leakage before training.

In [5]:
import csv
from pathlib import Path, PureWindowsPath

SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
split_rows = {}
for split_name in ("train", "validation", "test"):
    manifest = SPLITS_DIR / f"{split_name}.csv"
    with manifest.open(newline="", encoding="utf-8") as file:
        split_rows[split_name] = list(csv.DictReader(file))

all_rows = [row for rows in split_rows.values() for row in rows]
all_paths = [row["image_path"] for row in all_rows]
assert len(all_rows) == 12_614
assert len(set(path.casefold() for path in all_paths)) == 12_614
assert all(not Path(path).is_absolute() and not PureWindowsPath(path).is_absolute() for path in all_paths)
assert all(Path(path).suffix.casefold() != ".png" for path in all_paths)
assert all("groundtruth" not in path.casefold() for path in all_paths)
missing = [path for path in all_paths if not (PROJECT_ROOT / path).is_file()]
assert not missing, f"Missing dataset files (first five): {missing[:5]}"
print({name: len(rows) for name, rows in split_rows.items()})
print("Portable paths, unique membership, file existence, and mask exclusion: PASS")

{'train': 8830, 'validation': 1892, 'test': 1892}
Portable paths, unique membership, file existence, and mask exclusion: PASS


## Train the standard Autoencoder

This calls the existing reusable trainer with the required configuration. It prints train loss, validation loss, and elapsed time after every epoch; early stopping monitors validation loss.

In [6]:
import importlib
import sys
from argparse import Namespace

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import train_autoencoder
importlib.reload(train_autoencoder)  # Avoid stale modules after updating the checkout.
train = train_autoencoder.train

training_args = Namespace(
    splits_dir=PROJECT_ROOT / "data" / "splits",
    checkpoint_path=PROJECT_ROOT / "checkpoints" / "best_autoencoder.pth",
    history_path=PROJECT_ROOT / "results" / "ae_training_history.csv",
    curve_path=PROJECT_ROOT / "outputs" / "ae" / "ae_training_curve.png",
    grid_path=PROJECT_ROOT / "outputs" / "ae" / "trained_reconstruction_grid.png",
    image_size=128,
    batch_size=32,
    num_workers=0,
    learning_rate=0.001,
    max_epochs=20,
    patience=4,
    seed=42,
    smoke_test=False,
)
training_summary = train(training_args)

Epoch 01/20 | train=0.02793163 | validation=0.01481912 | seconds=38.3
Epoch 02/20 | train=0.01231302 | validation=0.01065025 | seconds=37.3
Epoch 03/20 | train=0.01007854 | validation=0.00909575 | seconds=36.6
Epoch 04/20 | train=0.00878554 | validation=0.00806273 | seconds=36.5
Epoch 05/20 | train=0.00774116 | validation=0.00747666 | seconds=36.8
Epoch 06/20 | train=0.00666634 | validation=0.00631346 | seconds=36.7
Epoch 07/20 | train=0.00621751 | validation=0.00600352 | seconds=36.7
Epoch 08/20 | train=0.00596403 | validation=0.00563378 | seconds=36.4
Epoch 09/20 | train=0.00568453 | validation=0.00541308 | seconds=36.4
Epoch 10/20 | train=0.00558237 | validation=0.00526524 | seconds=36.6
Epoch 11/20 | train=0.00533023 | validation=0.00526668 | seconds=36.4
Epoch 12/20 | train=0.00518601 | validation=0.00514239 | seconds=36.8
Epoch 13/20 | train=0.00507911 | validation=0.00490428 | seconds=37.1
Epoch 14/20 | train=0.00496039 | validation=0.00467447 | seconds=36.7
Epoch 15/20 | train=

## Final training summary and artifact check

The trainer has already reloaded the best-validation checkpoint before creating the reconstruction grid. Full test-set MSE/PSNR/SSIM evaluation remains intentionally deferred.

In [7]:
required_artifacts = [
    training_args.checkpoint_path,
    training_args.history_path,
    training_args.curve_path,
    training_args.grid_path,
]
assert all(path.is_file() for path in required_artifacts)

print("GPU:", GPU_NAME)
print("Epochs completed:", training_summary["epochs_completed"])
print("First train loss:", training_summary["first_train_loss"])
print("Final train loss:", training_summary["final_train_loss"])
print("Best validation loss:", training_summary["best_validation_loss"])
print("Best epoch:", training_summary["best_epoch"])
print("Early stopping triggered:", training_summary["early_stopping_triggered"])
print("Total training time (seconds):", training_summary["training_time_seconds"])
print("Best checkpoint path:", training_summary["checkpoint_path"])

GPU: Tesla T4
Epochs completed: 20
First train loss: 0.027931625757573145
Final train loss: 0.004299333967618636
Best validation loss: 0.004161450421205283
Best epoch: 20
Early stopping triggered: False
Total training time (seconds): 733.160548318
Best checkpoint path: /content/Digital-Evidence-GenAI/checkpoints/best_autoencoder.pth
